In [1]:
import pandas as pd
import re
import csv
import numpy as np

In [2]:
df1 = pd.read_csv("Baseline_EFs/rateperprofile_smoke_17031_2022_OZONETOXICSPMGHGMETALS_17031_07.csv",skiprows=2, low_memory=False)
df2 = pd.read_csv("Baseline_EFs/rateperprofile_smoke_aq_cb6_saprc_20240220_rates2022v1-2022-20240214_17031_07.csv",skiprows=2, low_memory=False)
df3 = pd.read_csv("ZeroPre2010s_EFs/rateperprofile_smoke_17031_2022_OZONETOXICSPMGHGMETALS_17031_07.csv",skiprows=2, low_memory=False)

In [3]:
def _veh_code_from_agg_scc(series):
    s = series.astype(str).str.zfill(10)
    return s.str[4:6]  # 0-based index: positions 4 and 5 are the 5th/6th digits

In [4]:
align_keys = [
    'MOVESScenarioID','yearID','FIPS','monthID','dayID','hourID',
    'agg_scc','temperature',
]

df1['MOVESScenarioID'] = df1['MOVESScenarioID'].astype(str).str.strip().str.casefold()
df2['MOVESScenarioID'] = df2['MOVESScenarioID'].astype(str).str.strip().str.casefold()
df3['MOVESScenarioID'] = df3['MOVESScenarioID'].astype(str).str.strip().str.casefold()

for k in align_keys:
    assert k in df1.columns, f"df1 missing key column: {k}"
    assert k in df2.columns, f"df2 missing key column: {k}"
    assert k in df3.columns, f"df3 missing key column: {k}"

In [5]:
df1.loc[(df1['agg_scc'].astype(str).str.zfill(10).str[4:6].isin(["52"]))]

,MOVESScenarioID,yearID,monthID,dayID,hourID,FIPS,agg_scc,temperature,THC_INV,BENZENE_INV,...,TOLUENE,XYLS,TOG_INV,VOC_INV,NONHAPTOG,EXH_NHTOG,EPM_NHTOG,EVP_NHTOG,RFL_NHTOG,ETHYLBENZ
192,rp_17031_2022_7_profm2022070001,2022,7,2,1,17031,2201520172,34.2,0.000641,0.000003,...,0.000098,0.000044,0.000687,0.000687,0.000408,0,0,0.000408,0,1.113371e-07
193,rp_17031_2022_7_profm2022070001,2022,7,2,2,17031,2201520172,30.3,0.009264,0.000038,...,0.001422,0.000637,0.009921,0.009921,0.005889,0,0,0.005889,0,1.608271e-06
194,rp_17031_2022_7_profm2022070001,2022,7,2,3,17031,2201520172,27.0,0.009267,0.000038,...,0.001423,0.000637,0.009925,0.009925,0.005891,0,0,0.005891,0,1.608897e-06
195,rp_17031_2022_7_profm2022070001,2022,7,2,4,17031,2201520172,24.3,0.009263,0.000038,...,0.001422,0.000637,0.009921,0.009921,0.005889,0,0,0.005889,0,1.608235e-06
196,rp_17031_2022_7_profm2022070001,2022,7,2,5,17031,2201520172,21.8,0.009255,0.000037,...,0.001421,0.000637,0.009912,0.009912,0.005884,0,0,0.005884,0,1.606822e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47371,rp_17031_2022_7_profm2022070066,2022,7,5,20,17031,2201520172,20.0,0.018418,0.000075,...,0.002828,0.001267,0.019726,0.019726,0.011709,0,0,0.011709,0,3.197614e-06
47372,rp_17031_2022_7_profm2022070066,2022,7,5,21,17031,2201520172,20.0,0.000492,0.000002,...,0.000076,0.000034,0.000527,0.000527,0.000313,0,0,0.000313,0,8.543162e-08
47373,rp_17031_2022_7_profm2022070066,2022,7,5,22,17031,2201520172,20.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0.000000,0,0.000000e+00
47374,rp_17031_2022_7_profm2022070066,2022,7,5,23,17031,2201520172,20.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0.000000,0,0.000000e+00


In [6]:
df2.loc[(df2['agg_scc'].astype(str).str.zfill(10).str[4:6].isin(["52"]))]

,MOVESScenarioID,yearID,monthID,dayID,hourID,FIPS,agg_scc,temperature,THC_INV,BENZENE_INV,...,TOLUENE,XYLS,TOG_INV,VOC_INV,NONHAPTOG,EXH_NHTOG,EPM_NHTOG,EVP_NHTOG,RFL_NHTOG,ETHYLBENZ
192,rp_17031_2022_7_profm2022070001,2022,7,2,1,17031,2201520172,34.2,0.000642,0.000003,...,0.000098,0.000044,0.000687,0.000687,0.000408,0,0,0.000408,0,1.113765e-07
193,rp_17031_2022_7_profm2022070001,2022,7,2,2,17031,2201520172,30.3,0.009264,0.000038,...,0.001422,0.000637,0.009922,0.009922,0.005889,0,0,0.005889,0,1.608359e-06
194,rp_17031_2022_7_profm2022070001,2022,7,2,3,17031,2201520172,27.0,0.009267,0.000038,...,0.001423,0.000637,0.009925,0.009925,0.005891,0,0,0.005891,0,1.608870e-06
195,rp_17031_2022_7_profm2022070001,2022,7,2,4,17031,2201520172,24.3,0.009265,0.000038,...,0.001423,0.000637,0.009923,0.009923,0.005890,0,0,0.005890,0,1.608460e-06
196,rp_17031_2022_7_profm2022070001,2022,7,2,5,17031,2201520172,21.8,0.009257,0.000037,...,0.001421,0.000637,0.009914,0.009914,0.005884,0,0,0.005884,0,1.607030e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47371,rp_17031_2022_7_profm2022070066,2022,7,5,20,17031,2201520172,20.0,0.018438,0.000075,...,0.002831,0.001268,0.019747,0.019747,0.011721,0,0,0.011721,0,3.200948e-06
47372,rp_17031_2022_7_profm2022070066,2022,7,5,21,17031,2201520172,20.0,0.000492,0.000002,...,0.000076,0.000034,0.000527,0.000527,0.000313,0,0,0.000313,0,8.548432e-08
47373,rp_17031_2022_7_profm2022070066,2022,7,5,22,17031,2201520172,20.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0.000000,0,0.000000e+00
47374,rp_17031_2022_7_profm2022070066,2022,7,5,23,17031,2201520172,20.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0.000000,0,0.000000e+00


In [7]:
df3.loc[(df3['agg_scc'].astype(str).str.zfill(10).str[4:6].isin(["52"]))]

,MOVESScenarioID,yearID,monthID,dayID,hourID,FIPS,agg_scc,temperature,THC_INV,BENZENE_INV,...,TOLUENE,XYLS,TOG_INV,VOC_INV,NONHAPTOG,EXH_NHTOG,EPM_NHTOG,EVP_NHTOG,RFL_NHTOG,ETHYLBENZ
192,rp_17031_2022_7_profm2022070001,2022,7,2,1,17031,2201520172,34.2,0.000132,5.337726e-07,...,0.000020,0.000009,0.000141,0.000141,0.000084,0,0,0.000084,0,2.287703e-08
193,rp_17031_2022_7_profm2022070001,2022,7,2,2,17031,2201520172,30.3,0.001907,7.725706e-06,...,0.000293,0.000131,0.002043,0.002043,0.001212,0,0,0.001212,0,3.311185e-07
194,rp_17031_2022_7_profm2022070001,2022,7,2,3,17031,2201520172,27.0,0.001909,7.732808e-06,...,0.000293,0.000131,0.002045,0.002045,0.001214,0,0,0.001214,0,3.314226e-07
195,rp_17031_2022_7_profm2022070001,2022,7,2,4,17031,2201520172,24.3,0.001907,7.725276e-06,...,0.000293,0.000131,0.002043,0.002043,0.001212,0,0,0.001212,0,3.311005e-07
196,rp_17031_2022_7_profm2022070001,2022,7,2,5,17031,2201520172,21.8,0.001903,7.709209e-06,...,0.000292,0.000131,0.002038,0.002038,0.001210,0,0,0.001210,0,3.304114e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47371,rp_17031_2022_7_profm2022070066,2022,7,5,20,17031,2201520172,20.0,0.003711,1.503035e-05,...,0.000570,0.000255,0.003974,0.003974,0.002359,0,0,0.002359,0,6.441897e-07
47372,rp_17031_2022_7_profm2022070066,2022,7,5,21,17031,2201520172,20.0,0.000100,4.039445e-07,...,0.000015,0.000007,0.000107,0.000107,0.000063,0,0,0.000063,0,1.731273e-08
47373,rp_17031_2022_7_profm2022070066,2022,7,5,22,17031,2201520172,20.0,0.000000,0.000000e+00,...,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0.000000,0,0.000000e+00
47374,rp_17031_2022_7_profm2022070066,2022,7,5,23,17031,2201520172,20.0,0.000000,0.000000e+00,...,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0.000000,0,0.000000e+00


<h1> Merge files 

In [8]:
merged_12 = pd.merge(
    df1, df2,
    on=align_keys,
    how='inner',
    suffixes=('_df1','_df2')
)

print("Merged df1 and df2:")
print("  df1 rows:", len(df1))
print("  df2 rows:", len(df2))
print("  merged rows (should be close to or equal to df1/df2):", len(merged_12))

numeric_cols_df1 = df1.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols_df2 = df2.select_dtypes(include=[np.number]).columns.tolist()

key_like_cols = set(align_keys)
pollutant_cols_df1 = [c for c in numeric_cols_df1 if c not in key_like_cols]
pollutant_cols_df2 = [c for c in numeric_cols_df2 if c not in key_like_cols]

pollutant_cols = sorted(list(set(pollutant_cols_df1).intersection(set(pollutant_cols_df2))))

print("Number of pollutant columns to compare/adjust:", len(pollutant_cols))

Merged df1 and df2:
  df1 rows: 47520
  df2 rows: 47520
  merged rows (should be close to or equal to df1/df2): 47400
Number of pollutant columns to compare/adjust: 16


In [9]:
for col in pollutant_cols:
    col_df1 = f"{col}_df1"
    col_df2 = f"{col}_df2"
    # Create a correction column representing (df1 - df2) for this pollutant
    merged_12[f"{col}_corr"] = merged_12[col_df1] - merged_12[col_df2]

for col in pollutant_cols:
    col_df1 = f"{col}_df1"
    col_df2 = f"{col}_df2"
    merged_12[f"{col}_diff"] = merged_12[col_df2] - merged_12[col_df1]  # df2 - df1 
    # Guard against divide-by-zero for percent difference:
    denom = merged_12[col_df1].replace(0, np.nan)
    merged_12[f"{col}_percdiff"] = (merged_12[col_df2] - merged_12[col_df1]) / denom * 100.0

diff_cols = [f"{c}_diff" for c in pollutant_cols]
if diff_cols:
    merged_12['max_abs_diff_value'] = merged_12[diff_cols].abs().max(axis=1)
    merged_12['max_abs_diff_column'] = merged_12[diff_cols].abs().idxmax(axis=1)

corr_cols = [f"{c}_corr" for c in pollutant_cols]
corr_with_keys = merged_12[align_keys + corr_cols].copy()

In [10]:
df3_adjusted = pd.merge(
    df3,
    corr_with_keys,
    on=align_keys,
    how='left'  # left: keep every df3 row; if a correction is missing, we'll treat it as 0
)

for c in corr_cols:
    df3_adjusted[c] = df3_adjusted[c].fillna(0.0)

for col in pollutant_cols:
    corr_col = f"{col}_corr"
    # df3_adjusted[col] is the original df3 value; subtract the correction to remove df1->df2 bias
    df3_adjusted[col] = df3_adjusted[col] - df3_adjusted[corr_col]
    df3_adjusted[col] = df3_adjusted[col].clip(lower=0.0)

df3_adjusted.drop(columns=corr_cols, inplace=True)
print("Applied corrections to df3 to create df3_adjusted.")

Applied corrections to df3 to create df3_adjusted.


In [11]:
df4 = df2.copy()

df1['veh_code'] = _veh_code_from_agg_scc(df1['agg_scc'])
df2['veh_code'] = _veh_code_from_agg_scc(df2['agg_scc'])
df3_adjusted['veh_code'] = _veh_code_from_agg_scc(df3_adjusted['agg_scc'])
df4['veh_code'] = _veh_code_from_agg_scc(df4['agg_scc'])

target_veh_codes = {'52','53','61','62'}

df4_indexed = df4.set_index(align_keys).sort_index()
df3a_indexed = df3_adjusted.set_index(align_keys).sort_index()

mask_target = df4['veh_code'].isin(target_veh_codes)
keys_to_replace = df4.loc[mask_target, align_keys]

print("Rows in df4 with vehicle codes to replace (52,53,61,62):", len(keys_to_replace))

idx_df4 = pd.MultiIndex.from_frame(keys_to_replace)
common_index = idx_df4.intersection(df3a_indexed.index)

print("Rows that actually match in df3_adjusted for replacement:", len(common_index))

Rows in df4 with vehicle codes to replace (52,53,61,62): 9504
Rows that actually match in df3_adjusted for replacement: 9480


In [12]:
shared_pollutant_cols = [c for c in pollutant_cols if c in df4_indexed.columns and c in df3a_indexed.columns]

df4_indexed.loc[common_index, shared_pollutant_cols] = df3a_indexed.loc[common_index, shared_pollutant_cols].values

df4 = df4_indexed.reset_index()
print("df4 has been created from df2 with selected rows replaced by df3_adjusted for vehicle codes 52, 53, 61, 62.")


df4 has been created from df2 with selected rows replaced by df3_adjusted for vehicle codes 52, 53, 61, 62.


In [13]:
df4.loc[(df4['agg_scc'].astype(str).str.zfill(10).str[4:6].isin(["52"]))]

,MOVESScenarioID,yearID,FIPS,monthID,dayID,hourID,agg_scc,temperature,THC_INV,BENZENE_INV,...,XYLS,TOG_INV,VOC_INV,NONHAPTOG,EXH_NHTOG,EPM_NHTOG,EVP_NHTOG,RFL_NHTOG,ETHYLBENZ,veh_code
8,rp_17031_2022_7_profm2022070001,2022,17031,7,2,1,2201520172,34.2,0.000132,5.347053e-07,...,0.000009,0.000141,0.000141,0.000084,0.0,0.0,0.000084,0.0,2.291645e-08,52
23,rp_17031_2022_7_profm2022070001,2022,17031,7,2,2,2201520172,30.3,0.001908,7.727856e-06,...,0.000131,0.002043,0.002043,0.001213,0.0,0.0,0.001213,0.0,3.312063e-07,52
38,rp_17031_2022_7_profm2022070001,2022,17031,7,2,3,2201520172,27.0,0.001909,7.732328e-06,...,0.000131,0.002044,0.002044,0.001213,0.0,0.0,0.001213,0.0,3.313954e-07,52
53,rp_17031_2022_7_profm2022070001,2022,17031,7,2,4,2201520172,24.3,0.001908,7.730537e-06,...,0.000131,0.002044,0.002044,0.001213,0.0,0.0,0.001213,0.0,3.313259e-07,52
68,rp_17031_2022_7_profm2022070001,2022,17031,7,2,5,2201520172,21.8,0.001904,7.713989e-06,...,0.000131,0.002040,0.002040,0.001211,0.0,0.0,0.001211,0.0,3.306187e-07,52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47453,rp_17031_2022_7_profm2022070066,2022,17031,7,5,20,2201520172,20.0,0.003730,1.510760e-05,...,0.000257,0.003994,0.003994,0.002371,0.0,0.0,0.002371,0.0,6.475239e-07,52
47468,rp_17031_2022_7_profm2022070066,2022,17031,7,5,21,2201520172,20.0,0.000100,4.051762e-07,...,0.000007,0.000107,0.000107,0.000064,0.0,0.0,0.000064,0.0,1.736543e-08,52
47483,rp_17031_2022_7_profm2022070066,2022,17031,7,5,22,2201520172,20.0,0.000000,0.000000e+00,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000e+00,52
47498,rp_17031_2022_7_profm2022070066,2022,17031,7,5,23,2201520172,20.0,0.000000,0.000000e+00,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000e+00,52


In [14]:
if 'veh_code' in df4.columns:
    df4 = df4.drop(columns=['veh_code'])

In [15]:
df2_source_path = "Baseline_EFs/rateperprofile_smoke_aq_cb6_saprc_20240220_rates2022v1-2022-20240214_17031_07.csv"

# read metadata lines (you already do this)
meta_lines = []
with open(df2_source_path, 'r', encoding='utf-8', newline='') as f_meta:
    meta_lines.append(next(f_meta))
    meta_lines.append(next(f_meta))


# read the data part as *strings* so we can see original formatting
df2_text = pd.read_csv(
    df2_source_path,
    skiprows=2,
    dtype=str,                # keep as text
    low_memory=False
)
assert len(df2_text) == len(df2), "Row count mismatch between df2_text and df2"

In [16]:
def format_like(original: str, value: float) -> str:
    """
    Format `value` to match the style of `original`:
    - respects scientific notation vs fixed vs integer
    - respects number of decimal places
    """
    if pd.isna(value):
        # If original is blank, keep it blank; otherwise just return original
        return "" if (original is None or str(original).strip() == "") else str(original)

    if original is None:
        original = ""
    original = str(original).strip()

    # If original was blank, fall back to a generic but non-evil format
    if original == "":
        # choose a reasonable default; adjust if you want
        return f"{value:.6f}".rstrip("0").rstrip(".") or "0"

    # Scientific notation?
    if "e" in original or "E" in original:
        lower = "e" in original and "E" not in original
        # Split off the base to count decimals
        base_part = re.split("[eE]", original, maxsplit=1)[0]
        if "." in base_part:
            dec = len(base_part.split(".")[-1])
        else:
            dec = 0
        fmt_spec = f".{dec}{'e' if lower else 'E'}"
        return format(value, fmt_spec)

    # Fixed-point decimal?
    if "." in original:
        dec = len(original.split(".")[-1])
        fmt_spec = f".{dec}f"
        return format(value, fmt_spec)

    # Otherwise, treat as integer
    return format(value, ".0f")


In [17]:
df2_keys = df2[align_keys].copy()
df2_keys["_row_id"] = np.arange(len(df2_keys))

# 2) Subset df4 to align_keys + pollutant columns you actually modified
#    `shared_pollutant_cols` was already defined earlier in your script.
df4_small = df4[align_keys + shared_pollutant_cols].copy()

# 3) Merge to get new numeric values for each row in df2, if available
df2_with_new = df2_keys.merge(df4_small, on=align_keys, how="left")

# 4) Restore original row order
df2_with_new = df2_with_new.sort_values("_row_id").set_index("_row_id")

# 5) Identify which rows are target vehicle codes (52, 53, 61, 62)
df2_with_new["veh_code"] = _veh_code_from_agg_scc(df2_with_new["agg_scc"])
mask_target = df2_with_new["veh_code"].isin(target_veh_codes)

# --- Build output strings DataFrame starting from original text ---

out_df = df2_text.copy()  # all columns as strings; correct order & header

# Ensure we only touch pollutant columns that exist in df2_text
pollutant_cols_to_write = [c for c in shared_pollutant_cols if c in out_df.columns]

# Loop row-by-row for target rows, column-by-column for pollutant columns
# and format each new numeric value to mimic the original string pattern.
for col in pollutant_cols_to_write:
    # new numeric values (may be NaN where df4 had no data — e.g. non-target rows)
    new_vals = df2_with_new[col].to_numpy()
    # current string values from original file
    current_strings = out_df[col].astype(str).to_numpy()

    for i in range(len(out_df)):
        if not mask_target.iloc[i]:
            # Not one of the rows we replaced; keep original string
            continue

        val = new_vals[i]
        # If we somehow don't have a new numeric value, keep original
        if pd.isna(val):
            continue

        orig_str = current_strings[i]
        current_strings[i] = format_like(orig_str, val)

    out_df[col] = current_strings


# --- Write out new CSV with original metadata lines and formatting preserved ---

out_path = "Combined_EFs/rateperprofile_smoke_aq_cb6_saprc_20240220_rates2022v1-2022-20240214_17031_07.csv"

with open(out_path, "w", encoding="utf-8", newline="") as f_out:
    # write the two original metadata lines verbatim
    for line in meta_lines:
        f_out.write(line if line.endswith("\n") else line + "\n")

    # write the data; everything is already correctly formatted as strings
    out_df.to_csv(
        f_out,
        index=False,
        quoting=csv.QUOTE_MINIMAL
    )